# S-DAM v2 — Colab Runner
Run cells in order. **Phase 2 runs before Phase 1 and Phase 3** (Phase 2 is the primary claim).

In [ ]:
# Cell 1 — Setup: clone repo and install requirements
!git clone https://github.com/Jaswanth-K1210/SDAM.git || echo 'repo already cloned'
%cd SDAM
!pip install -q -r requirements.txt

In [ ]:
# Cell 2 — GPU check
import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# Cell 3 — Integration test (Phase 0): must pass before any experiment
import torch
from sdam.model import SDAM
from sdam.utils import set_all_seeds

set_all_seeds(42)
model = SDAM(input_dim=128, beta=8.0)
model.reset_memory()
x = torch.randn(10, 128)
model.write(x)
x_hat = model.read(x)
assert x_hat.shape == x.shape, 'shape mismatch'
assert model.ssl.verify_orthogonality(), 'seeds not orthogonal'
assert model.mem.n_stored > 0, 'nothing written'
print('PASSED — diagnostics:', model.diagnostics())

In [ ]:
# Cell 4 — Phase 2 (interference) — PRIMARY EXPERIMENT, run first
!python experiments/phase2_interference.py

In [ ]:
# Cell 5 — Phase 1 (retrieval accuracy)
!python experiments/phase1_retrieval.py

In [ ]:
# Cell 6 — Phase 3 (curriculum ordering)
!python experiments/phase3_curriculum.py

In [ ]:
# Cell 7 — Print all JSON results
import glob, json
for path in sorted(glob.glob('results/*.json')):
    print('=' * 60)
    print(path)
    with open(path) as f:
        data = json.load(f)
    summary = {k: v for k, v in data.items() if not isinstance(v, list)}
    print(json.dumps(summary, indent=2))

In [ ]:
# Cell 8 — Display all PNG plots inline
import glob
from IPython.display import Image, display
for path in sorted(glob.glob('results/*.png')):
    print(path)
    display(Image(path))

In [ ]:
# Cell 9 — Backup: copy results to Drive, commit to GitHub
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/sinn_sdam_results
!cp -r results/* /content/drive/MyDrive/sinn_sdam_results/ 2>/dev/null || true
!cp -r results/* results_archive/ 2>/dev/null || true
!git add results_archive/ && git commit -m 'results: add experiment outputs' || echo 'nothing to commit'
# !git push   # uncomment after configuring credentials